# KinoCompass – смесь «популярность × Mult-VAE» на GPU

Гибрид шага 6б: популярные за последний год смешиваются с Mult-VAE (лучший вариант сетки шага 6а: «вкус» 400, β 0,5) по рангам с весом популярности 0,2; 0,4; 0,6; 0,8 (`model/hybrid.py`). Mult-VAE обучается один раз на все веса. Пачки для обучения собираются прямо на GPU – эпоха быстрее, чем в первом прогоне (25 с).

Ноутбук клонирует репозиторий и запускает **тот же** код проверки, что и в Docker (`model/evaluate.py`): те же люди, сценарии и метрики – смесь сравнивается с «популярность × ALS» парным тестом.

**Как запустить:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**, затем «Среда выполнения» → «Выполнить все». В конце скачается архив `hybrid_vae_<период>.zip` с отчётом – его нужно положить в `model/reports/`.

Время на T4 (бесплатный Colab): загрузка набора и сценарии – около 2 минут, обучение Mult-VAE – 8 минут (эпоха – 11,7 с вместо 25 с в первом прогоне; потери совпали до последней цифры), проверка четырёх смесей – 5,4 минуты; всего 15 минут. Вкладку не закрывать до конца.

In [1]:
# проверочная часть: validation – выбор настроек; test – только на шаге 6, один раз
PERIOD = "validation"

In [2]:
import torch

assert torch.cuda.is_available(), "нет GPU: Среда выполнения -> Сменить среду выполнения -> T4 GPU"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "ГБ")

Tesla T4 14.6 ГБ


In [3]:
# из /content: при повторном запуске в том же сеансе текущая папка – внутри удаляемого клона
%cd /content
!rm -rf /content/KinoCompass && git clone -q --depth 1 https://github.com/AlexTrav/KinoCompass.git /content/KinoCompass
!git -C /content/KinoCompass log --oneline -1
# те же версии, что в образе Docker (torch в Colab уже есть)
!pip install -q requests==2.34.2 pandas==3.1.0 pyarrow==26.0.1 scipy==1.18.1

/content
a8d76a7 (grafted, HEAD -> main, origin/main, origin/HEAD) Добавил гибрид «свежесть × вкус» и результаты Mult-VAE
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.


In [4]:
%cd /content/KinoCompass/model
!PYTHONPATH=/content/KinoCompass/data:/content/KinoCompass/research PERIOD={PERIOD} python -W ignore evaluate.py hybrid_vae

/content/KinoCompass/model
скачиваю https://files.grouplens.org/datasets/movielens/ml-32m.zip в память
  0 % (1 из 227 МБ)
  25 % (57 из 227 МБ)
  50 % (114 из 227 МБ)
  75 % (171 из 227 МБ)
  100 % (227 из 227 МБ)
  ratings: 32 000 204 строк
  tags: 2 000 072 строк
  movies: 87 585 строк
  links: 87 585 строк
кандидаты: candidates, 21249 фильмов
сценарии: {'cold5': {'people': 3912, 'median_inputs': 5, 'median_targets': 44}, 'cold10': {'people': 3903, 'median_inputs': 10, 'median_targets': 41}, 'warm': {'people': 5677, 'median_inputs': 308, 'median_targets': 8}}
cold5 popular_year ndcg@10 0.34536
cold10 popular_year ndcg@10 0.30037
warm popular_year ndcg@10 0.08778
vae_z400_b0.5: эпоха 1/40, потеря 1220.939, 11.2 с
vae_z400_b0.5: эпоха 10/40, потеря 1116.042, 11.7 с
vae_z400_b0.5: эпоха 20/40, потеря 1117.282, 11.7 с
vae_z400_b0.5: эпоха 30/40, потеря 1113.596, 11.7 с
vae_z400_b0.5: эпоха 40/40, потеря 1111.14, 11.7 с
cold5 hybrid_popular_year_vae_z400_b0.5_w0.2 ndcg@10 0.34995
cold10 

In [5]:
from google.colab import files

!cd /content/KinoCompass/model && zip -q -r /content/hybrid_vae_{PERIOD}.zip reports/hybrid_vae && ls -la reports/hybrid_vae
files.download(f"/content/hybrid_vae_{PERIOD}.zip")

total 160
drwxr-xr-x  2 root root   4096 Nov 11 09:36 .
drwxr-xr-x 11 root root   4096 Nov 11 09:36 ..
-rw-r--r--  1 root root  11476 Nov 11 09:36 validation.json
-rw-r--r--  1 root root 141225 Nov 11 09:36 validation_users.csv.gz
